# Inspect Condor SmokeTest output

Loads merged `.coffea` files produced by `sidm/scripts/merge_coffea_chunks_eos.py` from a Condor run and re-renders a few plots from `sidm/studies/signal_kinematics/lepton_kinematics_summary.ipynb` so we can sanity-check that Condor output matches what the notebook would produce.

In [ ]:
import sys, os, glob
import matplotlib.pyplot as plt
import coffea
from coffea.util import load

sidm_path = str(os.getcwd()).split('/sidm')[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities

utilities.set_plot_style()
%matplotlib inline

In [ ]:
# Point at the directory where you copied/merged the Condor outputs.
# After running `sidm/scripts/merge_coffea_chunks_eos.py`, this will contain SAMPLE.coffea files.
MERGED_DIR = '/uscms_data/d3/murtazas/SIDM/signal_merged'

merged_files = sorted(glob.glob(os.path.join(MERGED_DIR, '*.coffea')))
print(f'Found {len(merged_files)} merged files in {MERGED_DIR}:')
for f in merged_files:
    print(' ', os.path.basename(f))

In [ ]:
sample = '2Mu2E_500GeV_1p2GeV_1p9mm'
out = load(os.path.join(MERGED_DIR, f'{sample}.coffea'))

print('top-level keys :', list(out.keys()))
print('per-sample keys:', list(out['out'][sample].keys()))
print('hist names     :', list(out['out'][sample]['hists'].keys())[:10], '...')
print('processed      :', out.get('processed'))
print('exception      :', out.get('exception'))

# Numerical sanity
n_evt = out['out'][sample]['hists']['genBS_n'].sum().value
print(f'total events in genBS_n: {n_evt:.0f}')

## Reproduce a panel from `lepton_kinematics_summary.ipynb`

In [ ]:
channels = ['gen_leptons_final', 'gen_leptons_born']
h = out['out'][sample]['hists']

plt.subplots(2, 2, figsize=(24, 20))

plt.subplot(2, 2, 1)
utilities.plot(h['genBS_n'][channels[0], :], histtype='fill')
utilities.plot(h['genA_n'][channels[0], :])
plt.legend(['Truth BS', 'Truth As'])

plt.subplot(2, 2, 2)
utilities.plot(h['genMus_fromA_status'][channels[0], :])
utilities.plot(h['genEs_fromA_status'][channels[0], :])
utilities.plot(h['genMus_fromA_status'][channels[1], :])
utilities.plot(h['genEs_fromA_status'][channels[1], :])
plt.legend(['Final Muons', 'Final Electrons', 'Born Muons', 'Born Electrons'])

plt.subplot(2, 2, 3)
utilities.plot(h['genAs_pt'][channels[0], :], histtype='fill')
utilities.plot(h['genA_from_genMus_pt'][channels[0], :])
utilities.plot(h['genA_from_genEs_pt'][channels[0], :])
plt.legend(['Truth A', 'A from Mus', 'A from Es'])

plt.subplot(2, 2, 4)
utilities.plot(h['genAs_mass'][channels[0], :], histtype='fill')
utilities.plot(h['genA_from_genMus_mass'][channels[0], :])
utilities.plot(h['genA_from_genEs_mass'][channels[0], :])
plt.legend(['Truth A', 'A from Mus', 'A from Es'])

plt.show()